# Enrichment and functional validation mini-protocol
Evaluate prioritized genes and variants using pathway, genomic-annotation, and heritability-enrichment analyses.

#### Miniprotocol Timing
This is the total duration for the selected route; module-specific timings appear on their respective pages.
Timing: TBD

## Overview
This mini-protocol provides alternative functional follow-up routes. Step 1 calls [`gsea.ipynb`](https://statfungen.github.io/xqtl-protocol/code/enrichment/gsea.html), step 2 calls [`eoo_enrichment.ipynb`](https://statfungen.github.io/xqtl-protocol/code/enrichment/eoo_enrichment.html), steps 3–5 call [`gregor.ipynb`](https://statfungen.github.io/xqtl-protocol/code/enrichment/gregor.html), and steps 6–9 call [`sldsc_enrichment.ipynb`](https://statfungen.github.io/xqtl-protocol/code/enrichment/sldsc_enrichment.html).
Pathway analysis, enrichment-over-odds, GREGOR, and S-LDSC use different inputs and null models. Select the route matching the scientific question; only the numbered commands within the GREGOR and S-LDSC routes form ordered chains.

## Steps
Choose a route before running commands; the commands are not one mandatory chain.

| Analysis goal | Commands to run, in order | Inputs |
|---|---:|---|
| Test pathway and GO-term enrichment of gene groups | 1 | `tests/fixtures/gsea/protocol_example.pathway_genes.tsv` |
| Test annotation enrichment using variant-level odds | 2 | `tests/fixtures/eoo_enrichment/protocol_example.eoo_significant_variants.tsv.gz`; `input/enrichment/protocol_example.eoo_baseline_annotation.tsv` |
| Test overlap of index SNPs with genomic annotations using matched controls | 3 → 4 → 5 | `input/ld/protocol_example.index.snps.txt`; `tests/fixtures/gregor/protocol_example.bed.file.index`; `input/enrichment/protocol_example.gregor_ref` |
| Partition GWAS heritability across annotations | 6 → 7 → 8 | `input/enrichment/sldsc/colocboost_test_annotation_path.txt`; `input/enrichment/sldsc/reference_annotation0.txt`; `input/enrichment/sldsc/genome_reference_bfile.txt`; `tests/fixtures/sldsc_enrichment/sumstats_test_all.txt` |
| Re-meta-analyze a selected subset of S-LDSC traits | 8 → 9 | `output/sldsc_postprocess/protocol_example.sldsc_postprocess.rds`; `tests/fixtures/sldsc_enrichment/sumstats_test_category1.txt` |

Run only the route appropriate for the scientific question. GREGOR steps 3–5 and S-LDSC steps 6–8 are ordered workflows; step 9 is an optional follow-up.

### 1. [Test pathway and GO enrichment](https://statfungen.github.io/xqtl-protocol/code/enrichment/gsea.html)

**What it does:** Maps grouped genes to ENTREZ identifiers and tests KEGG and GO BP/CC/MF over-representation for each group.

**Timing**: TBD

In [ ]:
sos run pipeline/gsea.ipynb pathway_analysis \
    --genes_file tests/fixtures/gsea/protocol_example.pathway_genes.tsv \
    --name protocol_example \
    --pvalue_cutoff 1 --organism hsa \
    --cwd output/gsea

### 2. [Estimate enrichment over odds](https://statfungen.github.io/xqtl-protocol/code/enrichment/eoo_enrichment.html)

**What it does:** Estimates annotation odds ratios and enrichment with chromosome block-jackknife uncertainty.

**Timing**: TBD

In [ ]:
sos run pipeline/eoo_enrichment.ipynb enrichment \
    --significant_variants_path tests/fixtures/eoo_enrichment/protocol_example.eoo_significant_variants.tsv.gz \
    --baseline_anno_path tests/fixtures/eoo_enrichment/protocol_example.eoo_baseline_annotation.tsv.gz \
    --trait protocol_example \
    --annotation-name baseline \
    --cwd output/eoo_enrichment

### 3. [Create a GREGOR configuration](https://statfungen.github.io/xqtl-protocol/code/enrichment/gregor.html)

**What it does:** Writes the configuration connecting index SNPs, annotation BED files, population settings, and the GREGOR reference database.

**Timing**: TBD

In [ ]:
sos run pipeline/gregor.ipynb gregor_conf \
    --gregor_db <path/to/protocol_example.gregor_ref> \
    --index_snp_file tests/fixtures/gregor/index.snps.txt \
    --bed_file_index <path/to/protocol_example.bed.file.index> \
    --pop EUR \
    --cwd output/gregor

### 4. [Run GREGOR enrichment](https://statfungen.github.io/xqtl-protocol/code/enrichment/gregor.html)

**What it does:** Compares annotation overlap for index SNPs with overlap among LD- and frequency-matched control variants.

**Timing**: TBD

In [ ]:
sos run pipeline/gregor.ipynb gregor \
    --gregor_db <path/to/protocol_example.gregor_ref> \
    --index_snp_file tests/fixtures/gregor/index.snps.txt \
    --bed_file_index <path/to/protocol_example.bed.file.index> \
    --pop EUR \
    --cwd output/gregor

### 5. [Plot GREGOR Fisher enrichment](https://statfungen.github.io/xqtl-protocol/code/enrichment/gregor.html)

**What it does:** Compares annotation odds ratios from two GREGOR result sets.

**Timing**: TBD

In [ ]:
sos run pipeline/gregor.ipynb gregor_fisher_plot \
    --fisher1 <path/to/protocol_example.trait1_enrichment_results.txt> \
    --fisher2 <path/to/protocol_example.trait2_enrichment_results.txt> \
    --cwd output/gregor

### 6. [Build annotation LD scores](https://statfungen.github.io/xqtl-protocol/code/enrichment/sldsc_enrichment.html)

**What it does:** Converts genomic annotations into chromosome-level annotation and LD-score files for stratified LD-score regression.

**Timing**: TBD

In [ ]:
sos run pipeline/sldsc_enrichment.ipynb make_annotation_files_ldscore \
  --annotation_file <path/to/colocboost_test_annotation_path.txt> \
  --reference_anno_file <path/to/reference_annotation0.txt> \
  --genome_ref_file <path/to/genome_reference_bfile.txt> \
  --annotation_name protocol_example \
  --plink_name reference. --baseline_name annotations. --weight_name weights. \
  --cwd output/sldsc_ldscore -j 4


### 7. [Estimate stratified SNP heritability](https://statfungen.github.io/xqtl-protocol/code/enrichment/sldsc_enrichment.html)

**What it does:** Runs S-LDSC for each trait and annotation target to estimate annotation-specific heritability enrichment.

**Timing**: TBD

In [ ]:
sos run pipeline/sldsc_enrichment.ipynb get_heritability \
  --target_anno_dirs output/sldsc_ldscore/protocol_example_single_1 \
  --all_traits_file tests/fixtures/sldsc_enrichment/sumstats_test_all.txt \
  --sumstat_dir <path/to/sldsc> \
  --baseline_ld_dir <path/to/sldsc> \
  --weights_dir <path/to/sldsc> \
  --plink_name reference. --baseline_name annotations. --weight_name weights. \
  --annotation_name protocol_example \
  --maf_cutoff 0 --cwd output/sldsc_heritability -j 4



### 8. [Postprocess and meta-analyze S-LDSC results](https://statfungen.github.io/xqtl-protocol/code/enrichment/sldsc_enrichment.html)

**What it does:** Standardizes per-trait results and computes random-effects meta-analyses across traits.

**Timing**: TBD

In [ ]:
sos run pipeline/sldsc_enrichment.ipynb postprocess \
  --traits_file tests/fixtures/sldsc_enrichment/sumstats_test_all.txt \
  --heritability_cwd output/sldsc_heritability \
  --target_categories ANNOT_0 --target_categories_label protocol_example_annotation \
  --target_anno_dir output/sldsc_ldscore/protocol_example_single_1 \
  --annotation_name protocol_example \
  --maf_cutoff 0 --cwd output/sldsc_postprocess -j 4


### 9. [Re-meta-analyze a trait subset](https://statfungen.github.io/xqtl-protocol/code/enrichment/sldsc_enrichment.html)

**What it does:** Reuses postprocessed S-LDSC results to estimate enrichment for a selected subset without rerunning regression.

**Timing**: TBD

In [ ]:
sos run pipeline/sldsc_enrichment.ipynb meta_subset \
  --postprocess_rds output/sldsc_postprocess/protocol_example.sldsc_postprocess.rds \
  --subset_traits_file tests/fixtures/sldsc_enrichment/sumstats_test_category1.txt \
  --subset_name category1 --target_categories ANNOT_0 \
  --annotation_name protocol_example \
  --maf_cutoff 0 --cwd output/sldsc_postprocess -j 4

## Output Files

| Step | Relative path | Contents |
|---:|---|---|
| 1 | `output/gsea/pathway_analysis/protocol_example.combined_pathway_results.rds` | Standardized KEGG and GO enrichment results for all gene groups |
| 2 | `output/eoo_enrichment/enrichment/protocol_example.baseline.enrichment_results.rds` | Odds ratios, enrichment estimates, and block-jackknife uncertainty |
| 3 | `output/gregor/protocol_example.index.gregor.conf` | GREGOR configuration |
| 4 | `output/gregor/<name>_gregor_output/StatisticSummaryFile.txt` | Raw GREGOR overlap statistics |
| 4 | `output/gregor/<name>_variant_counts.txt` | Parsed annotation overlap counts |
| 4 | `output/gregor/<name>_enrichment_results.txt` | Fisher-test enrichment estimates |
| 5 | `output/gregor/<result1>_vs_<result2>_enrichment.pdf` | GREGOR odds-ratio comparison |
| 6 | `output/sldsc_ldscore/<annotation_name>/*.{annot.gz,l2.ldscore.parquet,l2.M}` | Annotation and LD-score products |
| 7 | `output/sldsc_heritability/<annotation_name>/<trait>.results` | Trait- and annotation-specific S-LDSC results |
| 8 | `output/sldsc_postprocess/<annotation_name>.sldsc_postprocess.rds` | Per-trait results and cross-trait meta-analysis |
| 9 | `output/sldsc_postprocess/<subset_name>*` | Trait-subset meta-analysis tables |

## Anticipated Results

Pathway analysis summarizes biological processes represented by prioritized genes. Enrichment-over-odds and GREGOR test whether prioritized variants overlap functional annotations more than expected under their respective background models. S-LDSC tests whether GWAS heritability is disproportionately concentrated in annotations while accounting for LD.

Interpret enrichment in light of the selected background, annotation coverage, population-matched reference data, multiple testing, and uncertainty. Enrichment supports functional relevance but does not by itself validate a causal gene, variant, or mechanism.

## Command interface

In [ ]:
sos run pipeline/gsea.ipynb -h

In [ ]:
sos run pipeline/eoo_enrichment.ipynb -h

In [ ]:
sos run pipeline/gregor.ipynb -h

In [ ]:
sos run pipeline/sldsc_enrichment.ipynb -h